# Analysis — `<your-usecase>`

Starter notebook for ad-hoc queries against your replica.

Prerequisites:
- `docker compose up -d` is running (GraphDB on :7201, Streamlit on :8501)
- Your replica's TTL files have been uploaded to the `workspace_demo` repository (or whatever you set `LOCAL_WORKSPACE` to in `.env`)

The notebook talks to the platform's Python backend via `pip install digicities`.
Until the platform publishes a PyPI release, install directly from git:

```bash
pip install git+https://github.com/uesl-empa/digicities-platform.git@v0.1.0
```

In [ ]:
import os
from pathlib import Path

os.environ.setdefault('GRAPHDB_URL', 'http://localhost:7201')

from backend.graphdb import GraphDBClient

client = GraphDBClient(token='local', selected_repo='workspace_demo')
client.auth_mode

## 1. Load a replica

Adjust `TTL_PATH` to point at the `.ttl` you want to query. The default loads everything in `ingestion/output/` into a single named graph.

In [ ]:
REPO_ROOT = Path('..').resolve()
OUTPUT_DIR = REPO_ROOT / 'ingestion' / 'output'
GRAPH_NAME = '<https://example.org/my_usecase/replica>'

for ttl in OUTPUT_DIR.glob('*.ttl'):
    client.upload_ttl(
        ttl_str=ttl.read_text(encoding='utf-8'),
        graph_name=GRAPH_NAME,
        replace_existing=(ttl == next(OUTPUT_DIR.glob('*.ttl'))),
    )
    print(f'uploaded {ttl.name}')

## 2. Run a saved query

Queries live in `queries/`. Use the example as a starting point.

In [ ]:
QUERIES_DIR = REPO_ROOT / 'queries'

query = (QUERIES_DIR / 'example_list_components.rq').read_text(encoding='utf-8')
client.sparql_api_query(
    f'WITH {GRAPH_NAME}\n' + query,
    out_format='df',
)

## 3. Your analysis

Add cells below. Save queries you'll re-use into `queries/` so they're tracked.